In [1]:
!pip install vllm lm-eval duckdb dagster dagster-webserver

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 155.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 M

In [6]:
!nohup python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --host 0.0.0.0 \
    --port 8000 > /tmp/vllm.log 2>&1 &

import time, subprocess

print("Polling for server readiness...")
for i in range(24):  # max 2 minutes
    result = subprocess.run(
        ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
         "http://localhost:8000/health"],
        capture_output=True, text=True
    )
    if result.stdout.strip() == "200":
        print(f"Server ready after ~{(i+1)*5}s")
        break
    print(f"  {i*5}s — status: {result.stdout.strip() or 'no response'}")
    time.sleep(5)
else:
    print("Server didn't start in time — check logs:")
    subprocess.run(["tail", "-20", "/tmp/vllm.log"])

Polling for server readiness...
  0s — status: 000
  5s — status: 000
  10s — status: 000
  15s — status: 000
  20s — status: 000
  25s — status: 000
  30s — status: 000
  35s — status: 000
  40s — status: 000
  45s — status: 000
  50s — status: 000
  55s — status: 000
  60s — status: 000
Server ready after ~70s


In [27]:
!npm install -g localtunnel -q
import re, subprocess, time

lt = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = lt.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    match = re.search(r"https?://\S+", line)
    if match:
        url = match.group(0)
        break

if not url:
    raise RuntimeError("localtunnel did not print a URL within 30s")

print("URL:", url)


⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 841ms
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇URL: https://funny-windows-relax.loca.lt
